# Notebook 4 – Authentification des signatures
**Projet Computer Vision IG.2405 – 2026**

Ce notebook couvre la reconnaissance des signatures manuscrites :
1. **Extraction** de la zone de signature
2. **Prétraitement** : rognage, binarisation Otsu, normalisation
3. **Descripteur** : HOG (Histogram of Oriented Gradients) + Moments de Hu
4. **Identification** : similarité cosinus par rapport à la base de signatures
5. **Évaluation** : taux d'identification correcte

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure

from utils.grid_decoder import normalize_page, extract_signature_roi, get_roi, ROI_SIGNATURE
from utils.form_aligner import deskew
from utils.signature_utils import (
    preprocess_signature, compute_signature_descriptor,
    cosine_similarity, SIG_SIZE, HOG_PIXELS, HOG_CELLS, HOG_ORIENT,
)

FORM_DIR = os.path.join('PROJECT 2026 -DATABASE-20260518', 'FORM1')

img_files = sorted([f for f in os.listdir(FORM_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'{len(img_files)} images de présence disponibles')

## 1. Extraction de la zone de signature

La signature est dans la zone `ROI_SIGNATURE = (30, 272, 372, 288)` de l'image normalisée.

In [ ]:
# Afficher les signatures de plusieurs étudiants
n_show = min(8, len(img_files))
sigs = []

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, fname in enumerate(img_files[:n_show]):
    img = cv2.imread(os.path.join(FORM_DIR, fname))
    img_ds = deskew(img)
    norm   = normalize_page(img_ds)
    sig    = extract_signature_roi(norm)
    sigs.append((fname, sig))

    axes[i].imshow(cv2.cvtColor(sig, cv2.COLOR_BGR2RGB))
    sid = fname.split('_')[-1].split('.')[0]
    axes[i].set_title(f'ID: {sid}', fontsize=8)
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Zones de signature extraites', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Prétraitement de la signature

Pipeline de normalisation :
1. Conversion en niveaux de gris
2. **Rognage automatique** (crop des bords blancs via composantes connexes)
3. **Binarisation Otsu** (encre = 255, fond = 0)
4. Redimensionnement vers `SIG_SIZE = (128, 64)` px

In [ ]:
# Prétraitement d'une signature exemple
_, sig_example = sigs[0]
fname_example  = sigs[0][0]

# Étapes détaillées
gray = cv2.cvtColor(sig_example, cv2.COLOR_BGR2GRAY)
_, bin_inv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Rognage
coords = cv2.findNonZero(bin_inv)
if coords is not None and len(coords) > 10:
    x, y, w, h = cv2.boundingRect(coords)
    margin = 4
    x = max(0, x - margin); y = max(0, y - margin)
    w = min(gray.shape[1] - x, w + 2*margin)
    h = min(gray.shape[0] - y, h + 2*margin)
    gray_cropped  = gray[y:y+h, x:x+w]
    bin_cropped   = bin_inv[y:y+h, x:x+w]
else:
    gray_cropped  = gray
    bin_cropped   = bin_inv

# Redimensionnement
sig_norm = cv2.resize(bin_cropped, SIG_SIZE)  # W, H

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

axes[0].imshow(cv2.cvtColor(sig_example, cv2.COLOR_BGR2RGB))
axes[0].set_title('Brut')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Niveaux de gris')
axes[1].axis('off')

axes[2].imshow(bin_inv, cmap='gray')
axes[2].set_title('Binarisation Otsu')
axes[2].axis('off')

axes[3].imshow(sig_norm, cmap='gray')
axes[3].set_title(f'Normalisé {SIG_SIZE[0]}×{SIG_SIZE[1]}px')
axes[3].axis('off')

plt.suptitle('Prétraitement de la signature', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Descripteur HOG

Le **HOG (Histogram of Oriented Gradients)** capture l'orientation des contours de la signature.

Pour chaque cellule de `HOG_PIXELS` pixels :
$$H_k = \sum_{(x,y) \in \text{cellule}} \mathbb{1}[\lfloor\phi(x,y)/\Delta\phi\rfloor = k]$$

où $\phi(x,y) = \arctan(G_y / G_x)$ est l'orientation du gradient.

Paramètres : `{HOG_ORIENT}` orientations, cellules `{HOG_PIXELS}`, blocs `{HOG_CELLS}`.

In [ ]:
print(f'HOG paramètres : orientations={HOG_ORIENT}, pixels/cellule={HOG_PIXELS}, cellules/bloc={HOG_CELLS}')

# Calculer HOG et visualiser
hog_feat, hog_img = hog(
    sig_norm,
    orientations=HOG_ORIENT,
    pixels_per_cell=HOG_PIXELS,
    cells_per_block=HOG_CELLS,
    block_norm='L2-Hys',
    feature_vector=True,
    visualize=True,
)
hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))

print(f'Taille du descripteur HOG : {hog_feat.shape[0]} dimensions')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sig_norm, cmap='gray')
axes[0].set_title('Signature normalisée')
axes[0].axis('off')

axes[1].imshow(hog_img_rescaled, cmap='hot')
axes[1].set_title(f'Visualisation HOG ({hog_feat.shape[0]} features)')
axes[1].axis('off')

plt.suptitle('Descripteur HOG', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Moments de Hu

Les **7 moments de Hu** sont des invariants géométriques (rotation, échelle, translation) :

$$h_1, h_2, \ldots, h_7$$

Transformation log pour réduire l'écart d'échelle : $\tilde{h}_i = -\text{sgn}(h_i) \log_{10}|h_i|$

In [ ]:
moments = cv2.moments(sig_norm)
hu = cv2.HuMoments(moments).flatten()
hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

print('Moments de Hu (bruts)  :', hu)
print('Moments de Hu (log)    :', hu_log.round(3))

plt.figure(figsize=(8, 3))
plt.bar(range(7), hu_log, color='steelblue', edgecolor='black')
plt.xlabel('Indice du moment (h1 à h7)')
plt.ylabel('Valeur (log)')
plt.title('Moments de Hu – signature normalisée')
plt.xticks(range(7), [f'h{i+1}' for i in range(7)])
plt.tight_layout()
plt.show()

## 5. Descripteur combiné et similarité cosinus

Le descripteur final combine HOG + Moments de Hu, normalisé en norme L2 :

$$\mathbf{d} = \frac{[\text{HOG}, \tilde{h}_1, \ldots, \tilde{h}_7]}{\|[\text{HOG}, \tilde{h}_1, \ldots, \tilde{h}_7]\|_2}$$

La similarité entre deux signatures :

$$\text{sim}(\mathbf{d}_q, \mathbf{d}_r) = \cos(\mathbf{d}_q, \mathbf{d}_r) = \frac{\mathbf{d}_q \cdot \mathbf{d}_r}{\|\mathbf{d}_q\|\,\|\mathbf{d}_r\|}$$

In [ ]:
# Calculer les descripteurs pour toutes les signatures disponibles
descriptors = []
student_ids = []

for fname, sig_img in sigs:
    sid = fname.split('_')[-1].split('.')[0]
    d   = compute_signature_descriptor(preprocess_signature(sig_img))
    descriptors.append(d)
    student_ids.append(sid)

print(f'Descripteurs calculés : {len(descriptors)} × {descriptors[0].shape[0]} dimensions')

# Matrice de similarité
n = len(descriptors)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(descriptors[i], descriptors[j])

plt.figure(figsize=(8, 6))
im = plt.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, label='Similarité cosinus')
plt.xticks(range(n), [f'{sid}' for sid in student_ids], rotation=45, ha='right', fontsize=7)
plt.yticks(range(n), [f'{sid}' for sid in student_ids], fontsize=7)
plt.title('Matrice de similarité des signatures\n(diagonale = auto-comparaison = 1)', fontsize=11)
plt.tight_layout()
plt.show()

# Auto-comparaison vs inter-étudiants
diag = [sim_matrix[i, i] for i in range(n)]
off_diag = [sim_matrix[i, j] for i in range(n) for j in range(n) if i != j]
print(f'\nSimilarité auto (diagonale) : {np.mean(diag):.3f} ± {np.std(diag):.3f}')
print(f'Similarité inter-étudiants  : {np.mean(off_diag):.3f} ± {np.std(off_diag):.3f}')

## 6. Construction de la base de descripteurs

En production, on charge la base de signatures (ZIP ou répertoire) et on pré-calcule
le descripteur **moyen** de chaque étudiant (robuste aux variations inter-signatures).

In [ ]:
# Construire une base de test à partir des photos disponibles
# (en production, cela vient du répertoire STUDENT_CLASS_SIGNATURES)

from utils.signature_utils import build_descriptor_db, identify_signature

# Créer une base avec une image par étudiant
raw_db = {}
for fname, sig_img in sigs:
    sid = fname.split('_')[-1].split('.')[0]
    gray_sig = cv2.cvtColor(sig_img, cv2.COLOR_BGR2GRAY) if len(sig_img.shape) == 3 else sig_img
    raw_db[sid] = [gray_sig]

desc_db = build_descriptor_db(raw_db)
print(f'Base de descripteurs : {len(desc_db)} étudiants')

# Tester l'identification (leave-one-out)
correct = 0
total   = 0

for fname, sig_img in sigs:
    true_id = fname.split('_')[-1].split('.')[0]
    pred_id, score = identify_signature(sig_img, desc_db, threshold=0.65)
    ok = (pred_id == true_id)
    if ok:
        correct += 1
    total += 1
    print(f'  {true_id} → prédit={pred_id} (score={score:.3f}) [{"OK" if ok else "ERREUR"}]')

print(f'\nPrécision (même base, sans leave-one-out) : {correct}/{total} = {correct/total:.1%}')
print('(Note : utiliser leave-one-out pour une éval. honnête sur données séparées)')

## 7. Analyse du seuil de décision

Le seuil $\tau$ contrôle le compromis faux-positifs / faux-négatifs.

- Si `score < τ` → signature **non reconnue** (colonne C vide)
- Si `score ≥ τ` → student_id reconnu

In [ ]:
thresholds = np.linspace(0.5, 0.95, 20)
precision_list, recall_list = [], []

for tau in thresholds:
    tp, fp, fn = 0, 0, 0
    for fname, sig_img in sigs:
        true_id = fname.split('_')[-1].split('.')[0]
        pred_id, score = identify_signature(sig_img, desc_db, threshold=tau)
        if pred_id == true_id:
            tp += 1
        elif pred_id is not None:
            fp += 1
        else:
            fn += 1
    precision_list.append(tp / (tp + fp) if (tp + fp) > 0 else 1.0)
    recall_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)

plt.figure(figsize=(8, 4))
plt.plot(thresholds, precision_list, 'b-o', markersize=4, label='Précision')
plt.plot(thresholds, recall_list,    'r-s', markersize=4, label='Rappel')
plt.axvline(0.72, color='gray', linestyle='--', label='Seuil actuel (0.72)')
plt.xlabel('Seuil τ')
plt.ylabel('Score')
plt.title('Précision / Rappel selon le seuil de décision')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Bilan

| Composante | Description |
|---|---|
| Prétraitement | Rognage auto + binarisation Otsu + resize 128×64 |
| HOG | Capture l'orientation des contours (N features) |
| Moments de Hu | Invariants géométriques (7 valeurs) |
| Similarité | Cosinus entre descripteurs L2-normalisés |
| Seuil | Calibré à 0.72 (à optimiser sur base de validation) |

**Prochaine étape** → `05_ocr_textes.ipynb` : lecture des textes imprimés et manuscrits.